# Transformer Model — Electricity Price Prediction (24h ahead)

**Architecture:** Transformer Encoder (Vaswani et al., 2017) — PyTorch `nn.TransformerEncoder`  
**Target:** Day-ahead price 24h into the future  
**Best result:** MAE ~26 EUR/MWh  

**Features (20):**
- Price history: y_scaled, lag_1h, lag_24h, lag_168h, ma_24h, ma_168h
- Temporal: hour, day_of_week, day_of_year, month, year
- Calendar: is_holiday, is_weekend
- Grid: generation, consumption, wind_onshore
- Weather: temperature_c, humidity_percent, cloud_cover_percent, shortwave_radiation_wm2

## 1. Imports

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import holidays

# Device detection — works on Mac (MPS), GPU servers (CUDA), and CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

os.makedirs('models', exist_ok=True)

PyTorch: 2.11.0
Device: mps


## 2. Load & Clean Data

In [ ]:
# --- Prices ---

df = pd.read_csv('raw_data/combined_energy_price_clean.csv', sep='\t', low_memory=False)
df['Sequence'] = df['Sequence'].astype(str).str.strip()

df_clean = df[df['Sequence'] == '2'][['DateTime(UTC)', 'Price[Currency/MWh]']].copy()

df_clean.columns = ['ds', 'y']
df_clean['ds'] = pd.to_datetime(df_clean['ds'])
df_clean = df_clean.dropna().sort_values('ds').reset_index(drop=True)
df_clean = df_clean.set_index('ds').resample('h').mean().reset_index()
df_clean = df_clean.dropna().reset_index(drop=True)

# Cap outliers — 2021-2022 energy crisis spikes
df_clean['y'] = df_clean['y'].clip(upper=500)

print(f'Prices: {df_clean.shape}')
print(f'Range: {df_clean["ds"].min()} → {df_clean["ds"].max()}')

Prices: (98974, 2)
Range: 2015-01-04 23:00:00 → 2026-04-21 21:00:00


## 3. Feature Engineering

In [3]:
df_feat = df_clean.copy()

# --- Temporal features (present moment t) ---
df_feat['hour']        = df_feat['ds'].dt.hour
df_feat['day_of_week'] = df_feat['ds'].dt.dayofweek
df_feat['day_of_year'] = df_feat['ds'].dt.dayofyear
df_feat['month']       = df_feat['ds'].dt.month
df_feat['year']        = df_feat['ds'].dt.year

# --- Calendar flags (present moment t) ---
de_holidays = holidays.Germany(years=range(2015, 2027))
df_feat['is_holiday'] = df_feat['ds'].dt.date.apply(lambda x: 1 if x in de_holidays else 0)
df_feat['is_weekend'] = df_feat['ds'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

# --- Price lags (known past values — no leakage) ---
df_feat['lag_1h']   = df_feat['y'].shift(1)
df_feat['lag_24h']  = df_feat['y'].shift(24)
df_feat['lag_168h'] = df_feat['y'].shift(168)   # 1 week
df_feat['ma_24h']   = df_feat['y'].rolling(24).mean()
df_feat['ma_168h']  = df_feat['y'].rolling(168).mean()

df_feat = df_feat.dropna().reset_index(drop=True)
print(f'After feature engineering: {df_feat.shape}')

After feature engineering: (98806, 14)


In [4]:
# --- Known future features (t+24) ---
# These are features we KNOW at prediction time for the future moment
df_feat['ds_target'] = df_feat['ds'] + pd.Timedelta(hours=24)

# Calendar of the moment we are predicting
df_feat['target_hour']        = df_feat['ds_target'].dt.hour
df_feat['target_day_of_week'] = df_feat['ds_target'].dt.dayofweek
df_feat['target_month']       = df_feat['ds_target'].dt.month
df_feat['target_is_holiday']  = df_feat['ds_target'].dt.date.apply(lambda x: 1 if x in de_holidays else 0)
df_feat['target_is_weekend']  = df_feat['ds_target'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

df_feat = df_feat.drop(columns=['ds_target'])

print(f'After known future features: {df_feat.shape}')

After known future features: (98806, 19)


In [5]:
# --- Generation & Consumption ---
df_gen = pd.read_csv('raw_data/final_generation_minimized.csv')
df_gen.columns = ['ds', 'generation', 'consumption']
df_gen['ds'] = pd.to_datetime(df_gen['ds'])
df_gen = df_gen.set_index('ds').resample('h').mean().reset_index()
df_feat = df_feat.merge(df_gen, on='ds', how='left')
df_feat['generation']  = df_feat['generation'].interpolate()
df_feat['consumption'] = df_feat['consumption'].interpolate()

# --- Weather ---
df_weather = pd.read_csv('raw_data/weather_avg.csv')
df_weather['ds'] = pd.to_datetime(df_weather['DateTime(UTC)'])
df_weather = df_weather.drop(columns='DateTime(UTC)').sort_values('ds').reset_index(drop=True)
df_feat = df_feat.merge(df_weather, on='ds', how='left')
weather_cols = ['temperature_c', 'humidity_percent', 'cloud_cover_percent', 'shortwave_radiation_wm2']
df_feat[weather_cols] = df_feat[weather_cols].interpolate()

# --- Wind onshore generation ---
df_wind = pd.read_csv('raw_data/wind_onshore.csv')
df_wind['ds'] = pd.to_datetime(df_wind['ds'])
df_feat = df_feat.merge(df_wind, on='ds', how='left')
# Fill missing with hourly mean (pre-2018 gap — DE-LU zone not available)
df_feat['wind_onshore'] = df_feat.groupby(df_feat['ds'].dt.hour)['wind_onshore'].transform(
    lambda x: x.fillna(x.mean())
)

df_feat = df_feat.dropna().reset_index(drop=True)
print(f'After merging all features: {df_feat.shape}')
print(f'Nulls: {df_feat.isnull().sum()[df_feat.isnull().sum() > 0].to_dict()}')

After merging all features: (98806, 26)
Nulls: {}


## 4. Scaling & Target

In [6]:
# Target: price 24h ahead
df_feat['target'] = df_feat['y'].shift(-24)
df_feat = df_feat.dropna().reset_index(drop=True)

# Price scaler — kept separate for inverse_transform at evaluation
scaler_price = MinMaxScaler()
df_feat['y_scaled']      = scaler_price.fit_transform(df_feat[['y']])
df_feat['target_scaled'] = scaler_price.transform(df_feat[['target']].values.reshape(-1, 1))

# All other features scaled together
other_cols = ['hour', 'day_of_week', 'day_of_year', 'month', 'year',
              'lag_1h', 'lag_24h', 'lag_168h', 'ma_24h', 'ma_168h',
              'generation', 'consumption', 'temperature_c', 'humidity_percent',
              'cloud_cover_percent', 'shortwave_radiation_wm2',
              'is_holiday', 'is_weekend', 'wind_onshore', 'target_hour',
              'target_day_of_week', 'target_month',
              'target_is_holiday', 'target_is_weekend']

scaler_other = MinMaxScaler()
df_feat[other_cols] = scaler_other.fit_transform(df_feat[other_cols])

feature_cols = ['y_scaled'] + other_cols
print(f'Total features: {len(feature_cols)}')

Total features: 25


/Users/javierinocentezabala/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


## 5. Sequences & Split

In [6]:
def create_sequences(data, target, seq_len=24):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(target[i+seq_len])
    return np.array(X), np.array(y)

values = df_feat[feature_cols].values
target = df_feat['target_scaled'].values

X, y = create_sequences(values, target, seq_len=24)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

X_train = torch.FloatTensor(X_train).to(device)
X_test  = torch.FloatTensor(X_test).to(device)
y_train = torch.FloatTensor(y_train).to(device)
y_test  = torch.FloatTensor(y_test).to(device)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

X_train: torch.Size([79006, 24, 20]) | X_test: torch.Size([19752, 24, 20])


## 6. Model

Transformer Encoder for Time Series  
Architecture based on: Vaswani et al. (2017) "Attention Is All You Need" — https://arxiv.org/abs/1706.03762  
Implementation: PyTorch `nn.TransformerEncoder` — Paszke et al. (2019)

In [7]:
class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_projection(x)
        x = self.transformer(x)
        x = self.fc(x[:, -1, :])
        return x.squeeze(-1)

n_features = len(feature_cols)
model = TransformerModel(input_size=n_features).to(device)
print(f'Model ready — {n_features} input features')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Model ready — 20 input features
Parameters: 563,713


## 7. Training (with checkpoint)

If the saved model exists, training is skipped.

In [ ]:
MODEL_PATH = 'models/transformer_V3.pth'

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f'Model loaded from {MODEL_PATH} — skipping training')
else:
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    dataset = TensorDataset(X_train, y_train)
    loader  = DataLoader(dataset, batch_size=256, shuffle=True)

    epochs    = 100
    best_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.6f}')

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), MODEL_PATH)

    print(f'Training done. Best loss: {best_loss:.6f}')
    print(f'Model saved to {MODEL_PATH}')

Epoch 10/100 — Loss: 0.001539
Epoch 20/100 — Loss: 0.001006
Epoch 30/100 — Loss: 0.000660
Epoch 40/100 — Loss: 0.000528


## 8. Evaluation

In [ ]:
model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(X_test), 1000):
        batch = X_test[i:i+1000]
        preds.append(model(batch).cpu().numpy())

y_pred_np = np.concatenate(preds)
y_test_np = y_test.cpu().numpy()

y_pred_real = scaler_price.inverse_transform(y_pred_np.reshape(-1, 1)).flatten()
y_test_real = scaler_price.inverse_transform(y_test_np.reshape(-1, 1)).flatten()

mae  = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print(f'MAE:  {mae:.2f} EUR/MWh')
print(f'RMSE: {rmse:.2f} EUR/MWh')

MAE:  28.21 EUR/MWh
RMSE: 38.68 EUR/MWh


## 9. Forecast vs Actual

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(y_test_real[:500], label='Actual', color='black', linewidth=0.8)
plt.plot(y_pred_real[:500], label='Transformer (24h ahead)', color='orange', linewidth=0.8)
plt.title(f'Transformer Forecast vs Actual — MAE: {mae:.2f} EUR/MWh')
plt.ylabel('Price (EUR/MWh)')
plt.legend()
plt.tight_layout()
plt.show()